# NB03: Figures and Visualizations

Standalone notebook that reads intermediate results from NB02 (CSV/parquet outputs) and generates publication-quality figures. Requires NO Spark access — can be run locally.

> **PENDING** — This notebook requires `data/hotspots_5grid.csv` and `data/biome_stratified_prevalence.csv` produced by NB10b (10b_spatial_analysis.ipynb). Run NB10a then NB10b first. The path bug (`PROJ = Path.cwd().parent`) has been fixed in the setup cell below.

## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set consistent style
sns.set_style('whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

# Paths — .parent needed because cwd() is the notebooks/ directory
PROJ = Path.cwd().parent
data_dir = PROJ / 'data'
fig_dir = PROJ / 'figures'
fig_dir.mkdir(exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Figure directory: {fig_dir}")

## Section 2: Load Intermediate Results from NB02

In [ ]:
# Load hotspot enrichment table (5° grid results)
hotspots = pd.read_csv(data_dir / 'hotspots_5grid.csv')
print(f"Loaded hotspot table: {len(hotspots)} significant hotspots")

# Load biome-stratified prevalence table
biome_table = pd.read_csv(data_dir / 'biome_stratified_prevalence.csv')
print(f"Loaded biome table: {len(biome_table)} biome categories")

# Optionally load full geospatial dataset for grid visualization
# (This allows us to recreate the grid scatter map)
try:
    grid_full = pd.read_csv(data_dir / 'grid_analysis_full.csv')  # if saved by NB02
    print(f"Loaded full grid analysis: {len(grid_full)} cells")
except FileNotFoundError:
    print("Note: full grid analysis not found — will only plot significant hotspots")
    grid_full = None

## Section 3: Global Hotspot Map

In [ ]:
# Create global 5° grid map highlighting significant hotspots
# (requires full grid from NB02; if unavailable, this will be a simplified scatter plot)

fig, ax = plt.subplots(figsize=(14, 7))

if grid_full is not None:
    # Full grid heatmap
    sc = ax.scatter(
        grid_full['lon_bin'] + 2.5,
        grid_full['lat_bin'] + 2.5,
        c=grid_full['prevalence'],
        cmap='RdYlGn_r',
        s=grid_full['n_total'] / 2,
        alpha=0.7,
        vmin=0,
        vmax=0.25,
        edgecolor='none'
    )
    cbar = plt.colorbar(sc, ax=ax, label='Metal resistance prevalence')
else:
    # Simplified: only plot hotspot centroids
    sc = ax.scatter(
        hotspots['lon_bin'] + 2.5,
        hotspots['lat_bin'] + 2.5,
        c=hotspots['prevalence'],
        cmap='RdYlGn_r',
        s=hotspots['n_total'] * 2,
        alpha=0.8,
        edgecolor='red',
        linewidth=1.5
    )
    cbar = plt.colorbar(sc, ax=ax, label='Metal resistance prevalence')
    ax.text(0.02, 0.98, 'Red-bordered circles = significant hotspots (OR>2, q<0.05)',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_xlabel('Longitude (°)', fontsize=12)
ax.set_ylabel('Latitude (°)', fontsize=12)
ax.set_title('Global Distribution of Metal Resistance Hotspots (5° Grid)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(fig_dir / 'nb03_global_hotspot_map.png', dpi=300, bbox_inches='tight')
print(f"Saved: {fig_dir / 'nb03_global_hotspot_map.png'}")
plt.close()

## Section 4: Biome Prevalence Bar Chart

In [ ]:
# Compute global prevalence for reference line
# (infer from biome data; it's the mean of all MAGs)
# For simplicity, compute as total resistant / total MAGs across biomes
total_n = biome_table['n'].sum()
total_resistant = (biome_table['n'] * biome_table['prevalence']).sum()
global_prev = total_resistant / total_n if total_n > 0 else biome_table['prevalence'].mean()

# Create bar chart
fig, ax = plt.subplots(figsize=(8, 5))

# Color bars by significance
colors = ['#d62728' if q < 0.05 else '#aec7e8' for q in biome_table['q']]

bars = ax.barh(
    biome_table['biome'],
    biome_table['prevalence'],
    color=colors,
    edgecolor='black',
    linewidth=1
)

# Reference line for global prevalence
ax.axvline(
    global_prev,
    color='black',
    linestyle='--',
    linewidth=2,
    label=f'Global prevalence = {global_prev:.3f}'
)

ax.set_xlabel('Metal Resistance Prevalence', fontsize=12, fontweight='bold')
ax.set_ylabel('Biome Category', fontsize=12, fontweight='bold')
ax.set_title('Metal Resistance Prevalence by Biome', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)

# Add value labels on bars
for i, (idx, row) in enumerate(biome_table.iterrows()):
    ax.text(
        row['prevalence'] + 0.002,
        i,
        f"{row['prevalence']:.3f} (n={int(row['n'])})",
        va='center',
        fontsize=9
    )

plt.tight_layout()
plt.savefig(fig_dir / 'nb03_biome_prevalence.png', dpi=300, bbox_inches='tight')
print(f"Saved: {fig_dir / 'nb03_biome_prevalence.png'}")
plt.close()

## Section 5: Supplemental Figures

In [ ]:
# Create a summary table figure for top 10 hotspots with key statistics
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('tight')
ax.axis('off')

# Prepare table data (top 10 hotspots)
top_hotspots = hotspots.head(10)[['lat_bin', 'lon_bin', 'n_total', 'prevalence', 'OR', 'q']].copy()
top_hotspots.columns = ['Latitude', 'Longitude', 'N MAGs', 'Prevalence', 'OR', 'q-value']
top_hotspots['Prevalence'] = top_hotspots['Prevalence'].apply(lambda x: f"{x:.1%}")
top_hotspots['OR'] = top_hotspots['OR'].apply(lambda x: f"{x:.2f}")
top_hotspots['q-value'] = top_hotspots['q-value'].apply(lambda x: f"{x:.1e}")

table = ax.table(
    cellText=top_hotspots.values,
    colLabels=top_hotspots.columns,
    cellLoc='center',
    loc='center',
    colWidths=[0.12, 0.12, 0.12, 0.12, 0.12, 0.15]
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Color header row
for i in range(len(top_hotspots.columns)):
    table[(0, i)].set_facecolor('#4472C4')
    table[(0, i)].set_text_props(weight='bold', color='white')

plt.title('Top 10 Metal Resistance Hotspots (5° Grid)', fontsize=14, fontweight='bold', pad=20)
plt.savefig(fig_dir / 'nb03_hotspot_summary_table.png', dpi=300, bbox_inches='tight')
print(f"Saved: {fig_dir / 'nb03_hotspot_summary_table.png'}")
plt.close()

In [ ]:
# Print summary of generated figures
print("\n=== FIGURES GENERATED ===")
print(f"1. Global Hotspot Map: nb03_global_hotspot_map.png")
print(f"2. Biome Prevalence Chart: nb03_biome_prevalence.png")
print(f"3. Hotspot Summary Table: nb03_hotspot_summary_table.png")
print(f"\nAll figures saved to: {fig_dir}")
print(f"Figure resolution: 300 dpi (publication-quality)")
print(f"Style: seaborn whitegrid + muted palette")